In [1]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
import pyrealsense2 as rs

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# stream rgb and depth from L515 and save it. Press S to save and Q to quit

import pyrealsense2 as rs
import numpy as np
import cv2
import time
import os
from datetime import datetime

def realsense_stream_with_save():
    # Create save directory if it doesn't exist
    save_dir = "captured_images"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"Created directory: {save_dir}")
    
    # Configure depth and color streams
    pipeline = rs.pipeline()
    config = rs.config()
    
    # Enable color stream with 1920x1080 resolution
    config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
    
    # Enable depth stream for L515
    config.enable_stream(rs.stream.depth, 1024, 768, rs.format.z16, 30)
    
    # Start streaming
    pipeline.start(config)
    
    # Counter for saved images
    save_counter = 0
    
    try:
        while True:
            # Wait for a coherent pair of frames
            frames = pipeline.wait_for_frames()
            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()
            
            if not color_frame:
                continue
            
            # Convert images to numpy arrays
            color_image = np.asanyarray(color_frame.get_data())
            if depth_frame:
                depth_image = np.asanyarray(depth_frame.get_data())
                # Apply colormap to depth for visualization
                depth_colormap = cv2.applyColorMap(
                    cv2.convertScaleAbs(depth_image, alpha=0.03), 
                    cv2.COLORMAP_JET
                )
            
            # Display the color image
            cv2.namedWindow('RealSense L515 - 1920x1080', cv2.WINDOW_NORMAL)
            
            # Add instructions to the image
            instructions = "Press 'S' to save, 'Q' to quit"
            cv2.putText(color_image, instructions, (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            cv2.imshow('RealSense L515 - 1920x1080', color_image)
            
            # Display depth image if available
            if depth_frame:
                cv2.namedWindow('RealSense L515 - Depth', cv2.WINDOW_NORMAL)
                cv2.imshow('RealSense L515 - Depth', depth_colormap)
            
            # Handle key presses
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q') or key == 27:  # 'q' or ESC
                break
            elif key == ord('s'):  # 's' to save image
                save_counter += 1
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]
                
                # Generate filenames
                color_filename = f"{save_dir}/color_{timestamp}_{save_counter:04d}.png"
                depth_filename = f"{save_dir}/depth_{timestamp}_{save_counter:04d}.png"
                
                # Save color image
                cv2.imwrite(color_filename, color_image)
                print(f"Saved color image: {color_filename}")
                
                # Save depth image if available
                if depth_frame:
                    cv2.imwrite(depth_filename, depth_colormap)
                    print(f"Saved depth image: {depth_filename}")
                
                # Show confirmation on screen
                confirmation = f"Saved image #{save_counter}"
                temp_image = color_image.copy()
                cv2.putText(temp_image, confirmation, (10, 70), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.imshow('RealSense L515 - 1920x1080', temp_image)
                cv2.waitKey(500)  # Show confirmation for 500ms
                
    finally:
        # Stop streaming
        pipeline.stop()
        cv2.destroyAllWindows()
        print(f"Total images saved: {save_counter}")
        print(f"Images saved in: {os.path.abspath(save_dir)}")

if __name__ == "__main__":
    realsense_stream_with_save()

In [ ]:
# read already saved image do camera calibration, back projection

import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt



#https://docs.opencv.org/3.4/dc/dbb/tutorial_py_calibration.html
 # termination criteria
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((8*6,3), np.float32) # 8,6 vertices are visible
objp[:,:2] = np.mgrid[0:8,0:6].T.reshape(-1,2)
#objp=objp*25 mm #scale by the square size 
 # Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.


images = glob.glob(r'*.png')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (8,6),None)
    # If found, add object points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)
        corners2=cv2.cornerSubPix(gray,corners,(3,3),(-1,-1),criteria)
        imgpoints.append(corners2)
        # Draw and display the corners
#       cv2.drawChessboardCorners(img, (9,6), corners2,ret)
#       cv2.imshow('img',img)
#       cv2.waitKey(500)
        # Naming a window
        cv2.namedWindow("Resized_Window", cv2.WINDOW_NORMAL)
        # Using resizeWindow()
        cv2.resizeWindow("Resized_Window", 1024, 786)
        # Displaying the image
        img_draw=cv2.drawChessboardCorners(img, (9,6), corners2,ret)
        cv2.imshow("Resized_Window", img_draw)
        cv2.waitKey(50)

cv2.destroyAllWindows()

objpoints=np.array(objpoints)
imgpoints=np.array(imgpoints)

ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1],None,None)
#mtx is intrinsic matrix; 
# rvecs is rotation matrix, 
# tvecs is translation matrix

np.savez(r'C.npz', mtx=mtx, dist=dist, rvecs=rvecs, tvecs=tvecs)  # save the paramteters
#ret=np.array(ret)
#mtx=np.array(mtx)
#rvecs=np.array(rvecs)
#tvecs=np.array(tvecs)
#dist=np.array(dist)

In [4]:
# read calibration matrix and reporject axis at origin

def draw(img, corners, imgpts):
    corner = (tuple(corners[0].ravel()))
    img = cv2.line(img, corner, tuple(imgpts[0].ravel()), (255,0,0), 3)
    img = cv2.line(img, corner, tuple(imgpts[1].ravel()), (0,255,0), 3)
    img = cv2.line(img, corner, tuple(imgpts[2].ravel()), (0,0,255), 3)
    return img

#Load previously saved data
# with np.load(r'C.npz') as X:
#     mtx, dist, rvecs, tvecs = [X[i] for i in ('mtx','dist','rvecs','tvecs')]

axis = np.float32([[2,0,0], [0,2,0], [0,0,-2]]).reshape(-1,3) # x-blue, y-green, z-red
#Axis points are points in 3D space for drawing the axis. 
# We draw axis of length 2 (units will be in terms of chess square size since we calibrated
# based on that size). So our X axis is drawn from (0,0,0) to (3,0,0), so for Y axis. 
# For Z axis, it is drawn from (0,0,0) to (0,0,-3). Negative denotes it is drawn towards
#  the camera.
 
images = glob.glob(r'*.png')


for fname in images:
#for i in range(len(objpoints)):
    #cv2.namedWindow("output", cv2.WINDOW_NORMAL)
    img = cv2.imread(fname)
    #imS = cv2.resize(img, (960, 540)) 
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (8,6),None)

    if ret == True:
        
        corners2 = cv2.cornerSubPix(gray,corners,(11,11),(-1,-1),criteria)
        #objpoints=np.array(objpoints)
        #imgpoints=np.array(imgpoints)

        # Find the rotation and translation vectors.
        #This function returns the rotation and the translation vectors that transform a 
        # 3D point expressed in the object coordinate frame to the camera coordinate
        ret1, rvec1, tvec1 = cv2.solvePnP(objp, corners2, mtx, dist)     #solvePnPRansac   , inlier
        


        imgpts, jac = cv2.projectPoints(axis, rvec1, tvec1, mtx, dist) #objp  axis -  
        #object point to image point

        corners2=corners2.astype(int)
        imgpts=imgpts.astype(int)
        
        # Naming a window
        cv2.namedWindow("Resized_Window", cv2.WINDOW_NORMAL)
        # Using resizeWindow()
        cv2.resizeWindow("Resized_Window", 1024, 786)
        # Displaying the image
        img = draw(img,corners2,imgpts)
        cv2.imshow("Resized_Window", img)
        cv2.waitKey(500)
        
        
cv2.destroyAllWindows()


#reprojection error lower the better
mean_error = 0
for i in range(len(objpoints)):
    imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2)/len(imgpoints2) 
    # ground truth, estimated, L2 / number of points in the image =54 (for one image)
    mean_error += error # mean error for all image points in all images
print( "total error: {}".format(mean_error/len(objpoints)) ) # normalize to number of images
        

total error: 0.09966041665673035


In [5]:
print ('Interinsic Matrix:\n{}'.format(mtx))

rvec1_jacobian = cv2.Rodrigues(rvec1) # convert Jac to Rod -- 3x3 matrix for rotation
rot_ext=rvec1_jacobian[0]

print ('Exterinsic Rotation Matrix:\n{}'.format(rot_ext))

trans_ext=tvec1
print ('Exterinsic Translation Vector:\n{}'.format(trans_ext))

calib_ext=np.hstack((rot_ext, trans_ext))
print ('Exterinsic matrix:\n{}'.format(calib_ext))

Interinsic Matrix:
[[1.60439276e+03 0.00000000e+00 1.07691640e+03]
 [0.00000000e+00 2.08826709e+03 4.92665585e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Exterinsic Rotation Matrix:
[[ 0.91284744 -0.40815665 -0.01084896]
 [ 0.2051823   0.43559652  0.87644503]
 [-0.3530011  -0.80228662  0.48137969]]
Exterinsic Translation Vector:
[[-4.24850226]
 [ 0.09357041]
 [26.49098039]]
Exterinsic matrix:
[[ 9.12847438e-01 -4.08156654e-01 -1.08489600e-02 -4.24850226e+00]
 [ 2.05182298e-01  4.35596520e-01  8.76445033e-01  9.35704058e-02]
 [-3.53001103e-01 -8.02286618e-01  4.81379688e-01  2.64909804e+01]]
